# GEMS G14 — GeoFaultFormer: local–global geological fault discovery

**Status: authored without execution. Nothing in this notebook has ever run.** The first run should stay in `AUDIT_ONLY`, then `VALIDATE_DATA`, and read `docs/g14/COLAB_RUNBOOK.md` and `docs/g14/UNTESTED_RISKS.md` first.

This notebook only orchestrates: models, losses, splits and evaluation live in `gems_bench/g14/`. No cell trains, predicts or writes a submission unless the run mode **and** its flag allow it.

| run_mode | allowed |
|---|---|
| `AUDIT_ONLY` | inventory of inputs and existing artifacts (read-only) |
| `VALIDATE_DATA` | + channel stack, contract check, folds, dataset preview |
| `TRAIN` (+ `train_enabled`) | + training of the selected experiment |
| `EVALUATE` (+ `inference_enabled`) | + held-out maps, official-style evaluation, T0 references |
| `SUBMIT` (+ `submission_enabled`, confirm token) | + one validated GeoTIFF; upload stays manual |

Offline DTI values are nested proxies on held-out *known* faults, never leaderboard scores.

## 2. Configuration

In [ ]:
CONFIG = {
    "repo": "mybenkhadda/GEMS---drivendata",
    "ref": "g14-geofaultformer",
    "drive_root": "/content/drive/MyDrive/GEMS_2",
    "local_root": "/content/gems_work",

    "run_mode": "AUDIT_ONLY",          # AUDIT_ONLY | VALIDATE_DATA | TRAIN | EVALUATE | SUBMIT
    "train_enabled": False,
    "inference_enabled": False,
    "submission_enabled": False,

    "device": "auto",
    "seed": 2026,
    "experiment": "T1_local_cnn",      # see gems_bench.g14.experiments.matrix()
}
# Optional nested overrides (validated; a typo raises with the closest valid key), e.g.
OVERRIDES = {
    # "split": {"backend": "g13", "protocol": "spatial_cv_embargo"},
    # "data": {"g13_features_dir": "/content/gems_g13_work/features", "build_g13_features_if_missing": False},
    # "train": {"epochs": 30, "batch_size": 8, "accumulation": 2},
}

## 3–5. Environment, Drive, repository checkout

In [ ]:
import os, sys, subprocess, importlib
from google.colab import drive
drive.mount('/content/drive')
CODE = '/content/GEMS'
if not os.path.exists(CODE + '/.git'):
    subprocess.run(['git', 'clone', '-q', '--depth', '1', '-b', CONFIG['ref'],
                    f"https://github.com/{CONFIG['repo']}.git", CODE], check=True)
else:
    subprocess.run(['git', '-C', CODE, 'fetch', '-q', '--depth', '1', 'origin', CONFIG['ref']], check=True)
    subprocess.run(['git', '-C', CODE, 'checkout', '-q', 'FETCH_HEAD'], check=True)
COMMIT = subprocess.run(['git', '-C', CODE, 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
need = [p for p in ('xgboost', 'lightgbm', 'rasterio', 'skimage', 'tabulate') if importlib.util.find_spec(p) is None]
if need:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + [{'skimage': 'scikit-image'}.get(p, p) for p in need], check=True)
if CODE not in sys.path:
    sys.path.insert(0, CODE)
os.chdir(CODE)
import warnings; warnings.filterwarnings('ignore')
print('commit', COMMIT)

In [ ]:
import json, numpy as np, pandas as pd
from IPython.display import display, Markdown
from gems_bench.g14 import G14_VERSION
from gems_bench.g14.config import G14Config
from gems_bench.g14.experiments import matrix
from gems_bench.g14.pipeline import G14Pipeline

base = G14Config.from_dict(CONFIG).updated(OVERRIDES) if OVERRIDES else G14Config.from_dict(CONFIG)
pipe = G14Pipeline(base)
print('G14', G14_VERSION, '| experiment', pipe.cfg.experiment, '| hash', pipe.cfg.semantic_hash(), '| mode', pipe.cfg.run_mode)
display(matrix()[['name', 'group', 'kind', 'control', 'hypothesis']])

## 6. Artifact inventory (read-only, any mode)
Official inputs (found by content), G13 feature store / splits / regimes / scoreboard, P6 W2 maps, P7 outputs.

In [ ]:
inv = pipe.inventory()
display(inv)

## 7. Data-contract validation (VALIDATE_DATA+)
Builds the label-free channel stack and its 200 m / 800 m pyramids once (reused by schema hash), then checks alignment, channel order, NaN handling, mask shapes and flip safety. Read every FAIL before training.

In [ ]:
if pipe.cfg.run_mode != 'AUDIT_ONLY':
    chk = pipe.validate_data()
    display(chk[chk.status != 'PASS']); print(chk.status.value_counts())
    display(pd.read_csv(pipe.shared / 'stack' / 'channel_decisions.csv'))
else:
    print('AUDIT_ONLY: skipped')

## 8. Spatial fold generation and leakage audit (VALIDATE_DATA+)
Backend `g13` (locked TEST) or `phase5` (units shared with P6 W2 / P7). Hard assertions: no training pixel in the held-out zone or TEST, no held-out truth in the known context, no supervised withheld system.

In [ ]:
if pipe.cfg.run_mode != 'AUDIT_ONLY':
    display(pipe.build_splits())
    root = pipe.shared / 'folds' / f"{pipe.cfg.split.backend}_{pipe.cfg.split.protocol}"
    display(pd.read_csv(root / 'system_crossing.csv'))
    print(json.loads((root / 'fold_manifest.json').read_text())['mask_digest'])

## 9. Multiscale dataset inspection (VALIDATE_DATA+)
Three aligned views per sample: local 12.8 km (100 m), regional 51.2 km (200 m), global 204.8 km (800 m).

In [ ]:
if pipe.cfg.run_mode != 'AUDIT_ONLY':
    import matplotlib.pyplot as plt
    s = pipe.preview(0, 2)[0]
    fig, ax = plt.subplots(1, 4, figsize=(16, 4))
    ax[0].imshow(s['local'][0]); ax[0].set_title('local ch0')
    ax[1].imshow(s['regional'][0]); ax[1].set_title('regional ch0')
    ax[2].imshow(s['global'][0]); ax[2].set_title('global ch0')
    ax[3].imshow(s['targets']['trace']); ax[3].set_title('visible faults (loss region only)')
    print({k: v.shape for k, v in s.items() if hasattr(v, 'shape')}, s['meta'])

## 10–11. Model initialization and loss configuration
Parameter counts and one forward pass on a preview batch (CPU or GPU; no training).

In [ ]:
if pipe.cfg.run_mode != 'AUDIT_ONLY':
    import torch
    from gems_bench.g14.multiscale import UnitInputs, collate
    from gems_bench.g14.model import build_model, count_parameters
    from gems_bench.g14.losses import CompositeLoss
    u = pipe.units()[0]
    inp = UnitInputs(pipe.stack, pipe.data, u, pipe.cfg)
    model = build_model(pipe.cfg, inp)
    display(pd.Series(count_parameters(model)))
    b = collate(pipe.preview(0, 2))
    with torch.no_grad():
        out = model(b)
    loss, logs = CompositeLoss(pipe.cfg)(out, b['targets'])
    print({k: tuple(v.shape) for k, v in out.items() if hasattr(v, 'shape')}, logs)
    print('loss weights', pipe.cfg.loss.weights, pipe.cfg.loss.aux_weights)

## 12. Training (TRAIN + train_enabled)
Trains every unit of the protocol (and optionally the final unit for submission). Resumes from `latest` checkpoints; refuses a checkpoint whose config or schema hash differs.

In [ ]:
TRAIN_FINAL_UNIT = False
if pipe.cfg.run_mode == 'TRAIN' and pipe.cfg.train_enabled:
    metas = pipe.train(include_final=TRAIN_FINAL_UNIT)
    display(pd.DataFrame([{k: m[k] for k in ('unit', 'best_epoch', 'best_inner_score', 'seconds')} for m in metas]))
else:
    print('training disabled')

## 13. Validation: honest held-out maps (inference_enabled)
Each unit's model predicts its own held-out zone only (OOF / discovery construction).

In [ ]:
if pipe.cfg.inference_enabled and pipe.cfg.run_mode in ('TRAIN', 'EVALUATE', 'SUBMIT'):
    maps = pipe.predict_heldout(); print(sorted(maps))

## 14 & 19. Ablation comparison and official-style evaluation (EVALUATE)
Official components (pixel-exact known mask), (representation, area) chosen on the other zones, area grid to 30 %. T0 references are scored on the same units when they exist for this backend.

In [ ]:
if pipe.cfg.run_mode in ('EVALUATE', 'SUBMIT'):
    cmp = pipe.evaluate(include_references=True)
    display(cmp)
    # cross-experiment comparison: every experiment folder's scorecards
    cards = [json.loads(p.read_text()) for p in (pipe.local / 'experiments').glob('*/evaluation/*__scorecard.json')]
    if cards: display(pd.DataFrame(cards).filter(regex='^(model|disc_|geo_)'))

## 15–16. Segmentation visualization and geological regime diagnostics
Training curves and modality-gate summaries from saved manifests; maps from saved held-out arrays. Gate weights are diagnostics, not physical causality.

In [ ]:
from gems_bench.g14 import viz
for m in sorted((pipe.exp_dir / 'manifests').glob('*_train.json')) if (pipe.exp_dir / 'manifests').exists() else []:
    j = json.loads(m.read_text()); display(viz.training_curves(j['history'], j['unit']))
reg = pipe._regimes() if pipe.cfg.run_mode != 'AUDIT_ONLY' else None
print('regimes available:', reg is not None)

## 17. Segment recovery (graph refinement, T6)
Rule-based, evidence-gated linking on one held-out map at its cross-fitted operating point; the refined map is re-scored with the same evaluator. Proximity alone never links.

In [ ]:
RUN_GRAPH = False
if RUN_GRAPH and pipe.cfg.run_mode in ('EVALUATE', 'SUBMIT'):
    from gems_bench.g14 import graph
    from gems_bench.g14.evaluation import score
    maps = pipe.model_maps()
    st = pipe.g13_store
    ev = {k: st.physical(n) for k, n in (('mag', 'magnetic:rtp:tdx'), ('grav', 'gravity:iso_grav_anom:tdx'),
                                       ('coh', 'structural:rtp:orient_persistence')) if st is not None and st.has(n)}
    card = json.loads((pipe.exp_dir / 'evaluation' / f'{pipe.cfg.experiment}__scorecard.json').read_text())
    refined = {}
    for (task, zone), s in maps.items():
        u = [x for x in pipe.system.eval_units((task,)) if x.zone == zone][0]
        from gems_bench.g14.evaluation import _binary
        key = 'disc' if task.startswith('disc') else 'geo'
        b = _binary(u, s, card[f'{key}_rep'], card[f'{key}_frac'])
        r = graph.refine(np.nan_to_num(u.crop(s)), b, {k: v[u.sl] for k, v in ev.items()})
        full = np.full(pipe.data.shape, np.nan, np.float32); full[u.sl] = r['raster']
        refined[(task, zone)] = full
    display(pd.DataFrame([score(pipe.cfg.experiment + '_graph', refined, pipe.system, pipe.cfg, pipe.exp_dir / 'evaluation')['row']]))

## 18. Ensemble calibration (T7)
Only members with honest maps on the same units; every weight for zone z is fitted on the other zones.

In [ ]:
RUN_ENSEMBLE = False
if RUN_ENSEMBLE and pipe.cfg.run_mode in ('EVALUATE', 'SUBMIT'):
    from gems_bench.g14 import ensemble as E
    members = {pipe.cfg.experiment: pipe.model_maps(), **pipe.reference_maps()}
    ranks = E.ranked_members(members, pipe.system)
    names = list(members)
    fused, choice = E.weighted_crossfit(ranks, names, pipe.system, pipe.cfg, pipe.cfg.eval.primary_task)
    display(choice)

## 20. Submission generation (SUBMIT + submission_enabled + confirm)
Requires the final unit trained (`TRAIN_FINAL_UNIT=True`), its full map, and the cross-fitted operating point. Writes `predicted_faults.tif` through the G13 validator. Upload by hand only.

In [ ]:
CONFIRM = ''   # type 'WRITE_SUBMISSION' to write the file
if pipe.cfg.run_mode == 'SUBMIT' and pipe.cfg.submission_enabled and CONFIRM == 'WRITE_SUBMISSION':
    if not (pipe.exp_dir / 'maps' / 'full.npy').exists():
        pipe.predict_final()
    card = json.loads((pipe.exp_dir / 'evaluation' / f'{pipe.cfg.experiment}__scorecard.json').read_text())
    key = 'disc' if pipe.cfg.eval.primary_task.startswith('disc') else 'geo'
    rec = pipe.submission(card[f'{key}_rep'], card[f'{key}_frac'], confirm=CONFIRM)
    print(rec['validation']['valid'], rec['validation']['checks_failed'], rec['validation']['path'])
else:
    print('submission disabled')

## 21–22. Artifact synchronization and final experiment report

In [ ]:
if pipe.cfg.run_mode != 'AUDIT_ONLY':
    print('files copied to Drive:', pipe.sync())
p = pipe.report(); display(Markdown(p.read_text()))